# Imports and initialization

In [83]:
import re

import logging
import os
from pathlib import Path
import pandas as pd

import spacy
from spacy import displacy
from spacy.tokens.doc import Doc

# defining paths for notebook
current_dir = Path(globals()["_dh"][0])

csv_folder = os.path.join(current_dir, r"csv")
train_csv = "train.csv"

# logging configuration
logging.basicConfig(level=logging.DEBUG)

## Reading csv

In [89]:
# We are not using title, but we'd like to, add it here
used_columns = ["id", "text", "rating"]

df = pd.read_csv(os.path.join(csv_folder, train_csv), usecols=used_columns)

text_test = df["text"][10]

df.head()

,id,text,rating
0,17578,"Por incrível que pareça, para uma bebida desti...",5
1,18658,"O readset pode até ser bom, mais tem outros fo...",1
2,28477,"Foi difícil terminar esse livro , demorou mese...",2
3,43638,"A bola é boa divertida, mas não é nem um pouco...",2
4,26130,Comprei errado! Não tenho leitor de e-books. Q...,1


# Cleaning text

In [ ]:
def clean_text(text):
    text = (
        text.lower()
    )  # NOTE: talvez seja interessante deixar alguns caracteres em maiúsculo

    # remove URLs
    text = re.sub(r"http\\S+|www\\S+", " ", text)

    # keeps letters and spaces
    text = re.sub(
        r"[^a-zà-úâêîôûãõç\\s]", " ", text
    )  # NOTE: está tirando os números também

    # removes extra spaces
    text = re.sub(r"\\s+", " ", text).strip()

    return text

In [ ]:
print(clean_text(text_test))

# Features

In [ ]:
# 1,3, 6, 7, 9, 12, 14, 16, 17, 18
#
# 19-23
#
# 24-31
#
# tabela 4 ignora
#
# 24
#
# 46
#
# tabela 6 twitter

## Part-Of-Speech (POS) features

Don't forget to `python -m spacy download pt_core_news_sm`
See https://universaldependencies.org/u/pos/ for information on Tags

In [36]:
nlp = spacy.load("pt_core_news_sm")

In [90]:
doc_test = nlp(text_test)

In [91]:
# Feature 1
def number_of_adj(doc: Doc):
    logging.debug("[number_of_adj]")

    adj_num = 0
    for token in doc:
        if token.pos_ == "ADJ":
            adj_num += 1

            logging.debug(token.text + "\t|" + token.pos_)
    return adj_num


print(doc_test, "\nNumber of adj:", number_of_adj(doc_test))

DEBUG:root:[number_of_adj]
DEBUG:root:benefício	|ADJ


Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe. 
Number of adj: 1


In [94]:
def number_of_adv(doc: Doc):
    adv_num = 0

    for token in doc:
        if token.pos_ == "ADJ":
            adv_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return adv_num


print(doc_test, "\nNumber of adv:", number_of_adv(doc_test))

DEBUG:root:benefício	|ADJ


Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe. 
Number of adv: 1


### spaCy testing

In [56]:
doc = nlp(text_test)
for token in doc:
    print(token.text, "\t\t|", token.pos_, token.dep_)

Não 		| ADV advmod
gostei 		| VERB ROOT
, 		| PUNCT punct
uma 		| DET det
página 		| NOUN obj
veio 		| VERB ccomp
rasgada 		| VERB xcomp
. 		| PUNCT punct


In [65]:
displacy.render(doc, style="dep", options={"compact": "True"})